In [7]:
import pandas as pd
import numpy as np
import logging
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

BASE = r"C:\Users\Julian\Documents\SeptimoSemestre\Ingenieria de Datos\SkillScope"

In [8]:
logger.info("Cargando dataset completo...")

df_jobs = pd.read_csv(
    f"{BASE}/data/raw/linkedin_job_postings.csv",
    usecols=['job_link', 'job_title', 'company', 'job_location',
             'first_seen', 'search_city', 'search_country',
             'search_position', 'job_level', 'job_type'],
    dtype=str
)

logger.info(f"Total registros: {df_jobs.shape}")
print(df_jobs['search_country'].value_counts().head(10))

2026-03-04 10:56:23,322 - INFO - Cargando dataset completo...
2026-03-04 10:56:33,680 - INFO - Total registros: (1348454, 10)


search_country
United States     1149342
United Kingdom     113421
Canada              55972
Australia           29719
Name: count, dtype: int64


In [9]:
# ── TRANSFORM ────────────────────────────────────────────────

# 1. Filtrar solo registros de US (el más completo)
df_jobs_clean = df_jobs[df_jobs['search_country'] == 'United States'].copy()
logger.info(f"Registros United States: {df_jobs_clean.shape}")

# 2. Convertir fechas
df_jobs_clean['first_seen'] = pd.to_datetime(df_jobs_clean['first_seen'])

# 3. Agregar columna week (lunes de cada semana)
df_jobs_clean['week'] = df_jobs_clean['first_seen'] - pd.to_timedelta(
    df_jobs_clean['first_seen'].dt.dayofweek, unit='D'
)
df_jobs_clean['week'] = df_jobs_clean['week'].dt.date

# 4. Limpiar texto
df_jobs_clean['job_title']  = df_jobs_clean['job_title'].str.strip()
df_jobs_clean['company']    = df_jobs_clean['company'].str.strip()
df_jobs_clean['job_level']  = df_jobs_clean['job_level'].str.strip()
df_jobs_clean['job_type']   = df_jobs_clean['job_type'].str.strip()

# 5. Eliminar duplicados
before = len(df_jobs_clean)
df_jobs_clean = df_jobs_clean.drop_duplicates(subset='job_link')
logger.info(f"Duplicados eliminados: {before - len(df_jobs_clean)}")

# 6. Verificar resultado
logger.info(f"JOBS limpio final: {df_jobs_clean.shape}")
display(df_jobs_clean.head(3))
display(df_jobs_clean.dtypes)

2026-03-04 11:02:36,736 - INFO - Registros United States: (1149342, 10)
2026-03-04 11:02:39,925 - INFO - Duplicados eliminados: 0
2026-03-04 11:02:39,927 - INFO - JOBS limpio final: (1149342, 11)


,job_link,job_title,company,job_location,first_seen,search_city,search_country,search_position,job_level,job_type,week
0,https://www.linkedin.com/jobs/view/account-exe...,Account Executive - Dispensing (NorCal/Norther...,BD,"San Diego, CA",2024-01-15,Coronado,United States,Color Maker,Mid senior,Onsite,2024-01-15
1,https://www.linkedin.com/jobs/view/registered-...,Registered Nurse - RN Care Manager,Trinity Health MI,"Norton Shores, MI",2024-01-14,Grand Haven,United States,Director Nursing Service,Mid senior,Onsite,2024-01-08
2,https://www.linkedin.com/jobs/view/restaurant-...,RESTAURANT SUPERVISOR - THE FORKLIFT,Wasatch Adaptive Sports,"Sandy, UT",2024-01-14,Tooele,United States,Stand-In,Mid senior,Onsite,2024-01-08


job_link                   object
job_title                  object
company                    object
job_location               object
first_seen         datetime64[ns]
search_city                object
search_country             object
search_position            object
job_level                  object
job_type                   object
week                       object
dtype: object

In [10]:
# ── TRANSFORM SKILLS ─────────────────────────────────────────

# Cargar skills completo
df_skills = pd.read_csv(
    f"{BASE}/data/raw/job_skills.csv",
    dtype=str
)
logger.info(f"Skills total: {df_skills.shape}")

# Eliminar nulos
df_skills_clean = df_skills.dropna(subset=['job_skills']).copy()

# Filtrar solo jobs que están en df_jobs_clean (United States)
jobs_us = set(df_jobs_clean['job_link'])
df_skills_clean = df_skills_clean[df_skills_clean['job_link'].isin(jobs_us)]
logger.info(f"Skills filtradas a US: {df_skills_clean.shape}")

# Explotar las skills — cada skill en su propia fila
df_skills_exploded = df_skills_clean.copy()
df_skills_exploded['job_skills'] = df_skills_exploded['job_skills'].str.split(', ')
df_skills_exploded = df_skills_exploded.explode('job_skills')
df_skills_exploded['job_skills'] = df_skills_exploded['job_skills'].str.strip()
df_skills_exploded = df_skills_exploded[df_skills_exploded['job_skills'] != '']

logger.info(f"Skills explotadas: {df_skills_exploded.shape}")
display(df_skills_exploded.head(5))

2026-03-04 11:03:44,117 - INFO - Skills total: (1296381, 2)
2026-03-04 11:03:47,303 - INFO - Skills filtradas a US: (1103611, 2)
2026-03-04 11:04:12,368 - INFO - Skills explotadas: (23302387, 2)


,job_link,job_skills
0,https://www.linkedin.com/jobs/view/housekeeper...,Building Custodial Services
0,https://www.linkedin.com/jobs/view/housekeeper...,Cleaning
0,https://www.linkedin.com/jobs/view/housekeeper...,Janitorial Services
0,https://www.linkedin.com/jobs/view/housekeeper...,Materials Handling
0,https://www.linkedin.com/jobs/view/housekeeper...,Housekeeping


In [11]:
import duckdb

# ── LOAD ─────────────────────────────────────────────────────
DB_PATH = f"{BASE}/data/jobs.duckdb"
con = duckdb.connect(DB_PATH)

# Crear tablas
con.execute("""
    CREATE TABLE IF NOT EXISTS jobs (
        job_link        VARCHAR PRIMARY KEY,
        job_title       VARCHAR,
        company         VARCHAR,
        job_location    VARCHAR,
        first_seen      DATE,
        search_city     VARCHAR,
        search_country  VARCHAR,
        search_position VARCHAR,
        job_level       VARCHAR,
        job_type        VARCHAR,
        week            DATE
    )
""")

con.execute("""
    CREATE TABLE IF NOT EXISTS job_skills (
        job_link    VARCHAR,
        skill       VARCHAR,
        PRIMARY KEY (job_link, skill)
    )
""")

logger.info("Tablas creadas.")

# Insertar jobs
con.execute("INSERT OR IGNORE INTO jobs SELECT * FROM df_jobs_clean")
total_jobs = con.execute("SELECT COUNT(*) FROM jobs").fetchone()[0]
logger.info(f"Jobs cargados: {total_jobs:,}")

# Insertar skills
con.execute("INSERT OR IGNORE INTO job_skills SELECT * FROM df_skills_exploded")
total_skills = con.execute("SELECT COUNT(*) FROM job_skills").fetchone()[0]
logger.info(f"Skills cargadas: {total_skills:,}")

con.close()
logger.info(f"Base de datos guardada en: {DB_PATH}")

2026-03-04 11:06:05,049 - INFO - Tablas creadas.
2026-03-04 11:06:27,618 - INFO - Jobs cargados: 1,149,342
2026-03-04 11:15:13,018 - INFO - Skills cargadas: 23,180,623
2026-03-04 11:15:14,515 - INFO - Base de datos guardada en: C:\Users\Julian\Documents\SeptimoSemestre\Ingenieria de Datos\SkillScope/data/jobs.duckdb


In [2]:
import duckdb

BASE = r"C:\Users\Julian\Documents\SeptimoSemestre\Ingenieria de Datos\SkillScope"
DB_PATH = f"{BASE}/data/jobs.duckdb"

con = duckdb.connect(DB_PATH)

print("=== Top 10 Skills más demandadas ===")
display(con.execute("""
    SELECT skill, COUNT(*) AS menciones
    FROM job_skills
    GROUP BY skill
    ORDER BY menciones DESC
    LIMIT 10
""").df())

print("\n=== Jobs por nivel ===")
display(con.execute("""
    SELECT job_level, COUNT(*) AS total
    FROM jobs
    GROUP BY job_level
    ORDER BY total DESC
""").df())

print("\n=== Jobs por tipo ===")
display(con.execute("""
    SELECT job_type, COUNT(*) AS total
    FROM jobs
    GROUP BY job_type
    ORDER BY total DESC
""").df())

con.close()

=== Top 10 Skills más demandadas ===


,skill,menciones
0,Communication,308587
1,Teamwork,191454
2,Leadership,153541
3,Customer service,147593
4,Communication skills,97795
5,Customer Service,94973
6,Problem Solving,83554
7,Nursing,80043
8,Sales,78646
9,Problemsolving,78176



=== Jobs por nivel ===


,job_level,total
0,Mid senior,1024462
1,Associate,124880



=== Jobs por tipo ===


,job_type,total
0,Onsite,1140663
1,Hybrid,5039
2,Remote,3640
